<a href="https://colab.research.google.com/github/41371231h-netizen/Programming-Language/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2_ipynb%E3%80%8D4_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [10]:
!pip install -q google-generativeai

In [11]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [12]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [13]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1D6KLSFjG3UpTdh4_ZM7MT2Dc94DrRc6dPa2ZRCjAZ4E/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [14]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash-lite'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to make predictions or decisions.


### 定義 AI 摘要函式

In [15]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [22]:
import matplotlib.pyplot as plt
import pandas as pd

def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet、生成 AI 摘要並繪製圖表。
    """
    global _gc, _ws

    if _ws is None:
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            return "Google Sheet 連線失敗，請檢查權限設定。", "", None

    # 過濾空資料
    valid_grade_data = [row for row in grade_data if row[0].strip() and row[1].strip()]
    if not valid_grade_data:
        return "請輸入至少一科有效的科目與成績。", "", None

    new_rows = []
    today = datetime.now().strftime('%Y-%m-%d')

    subjects = []
    scores = []

    try:
        for subject, grade_str in valid_grade_data:
            grade = int(grade_str)
            new_rows.append([today, subject, grade])
            subjects.append(subject)
            scores.append(grade)
    except ValueError:
        return "成績欄位必須為數字，請重新檢查。", "", None

    # 繪製圖表
    plt.rcParams['font.sans-serif'] = ['Liberation Sans'] # Colab 預設字體
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(subjects, scores, color='skyblue')
    ax.set_title("Student Performance")
    ax.set_xlabel("Subjects")
    ax.set_ylabel("Scores")
    ax.set_ylim(0, 100)
    for i, v in enumerate(scores):
        ax.text(i, v + 2, str(v), ha='center')

    try:
        # 1. 批次寫入學生成績
        _ws.append_rows(new_rows)

        # 2. 獲取 AI 摘要
        summary = get_ai_summary(new_rows)

        # 3. 寫入 AI 摘要
        summary_row = [today, "AI 總結摘要", summary]
        _ws.append_row(summary_row)

        return "資料、圖表與 AI 摘要已成功同步！", summary, fig
    except Exception as e:
        error_msg = f"處理過程中發生錯誤：{e}"
        return error_msg, "", None

In [17]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")

--- 正在進行 Google Sheet 身份驗證和連線... ---
--- Google Sheet 連線成功。---

--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，這是一份根據您提供的學生成績所整理的簡單摘要與常見迷思整理：

## 學生成績摘要與迷思整理

**日期：** 2026-04-22

**科目表現摘要：**

*   **國文：** 成績為 85 分，表現穩健。
*   **數學：** 成績為 78 分，有進步空間。
*   **英文：** 成績為 92 分，表現優異。

**總結：**

從這份資料來看，該學生在英文科目上展現了非常出色的表現，國文也保持在一個良好的水平。數學科目雖然有分數，但相較於其他科目，可能還有加強的潛力。

---

### 常見學習迷思整理（針對此份成績單）

1.  **迷思一：數學分數低就代表「數學不好」**
    *   **事實：** 78 分是一個「及格」的成績，表示學生對數學的基本概念是有掌握的，只是可能在某些特定單元、解題技巧或練習量上還有提升的空間。不能直接斷定為「數學不好」。

2.  **迷思二：英文分數高代表「英文很厲害」**
    *   **事實：** 92 分確實是優異的表現，代表學生在英文的聽、說、讀、寫（或本次考試所涵蓋的範圍）都有相當好的掌握。但「厲害」是個相對概念，與其他學生的比較，或是與未來的學習目標相比，其程度仍有差異。

3.  **迷思三：科目之間「表現好壞」是固定的，無法改變**
    *   **事實：** 學習成效會受到學習方法、投入時間、興趣、教學方式等多重因素影響。這次數學分數的差異，並不代表未來也一定會如此。透過針對性的學習策略，數學成績是可以提升的。

4.  **迷思四：只要分數達到標準，就代表「學會了」**
    *   **事實：** 分數是學習成果的一種量化指標，但真正的「學會」更包含理解概念、融會貫通、並能將知識應用於不同情境的能力。一次的考試分數，不

定義 Gradio 處理函式

In [23]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 🎓 學生表現分析與視覺化工具")
    gr.Markdown("輸入成績後，系統將自動同步至 Google Sheet、生成 AI 建議並顯示統計圖表。")

    with gr.Row():
        with gr.Column(scale=1):
            grade_input = gr.Dataframe(
                headers=["科目", "成績"],
                value=[["", ""]],
                type="array",
                row_count=1,
                col_count=(2, "fixed"),
                label="成績輸入表",
                interactive=True
            )
            submit_button = gr.Button("🚀 提交並分析", variant="primary")
            sheet_output = gr.Textbox(label="📡 系統狀態", placeholder="等待提交...")

        with gr.Column(scale=1):
            plot_output = gr.Plot(label="📊 成績分佈圖")
            summary_output = gr.Textbox(label="🤖 AI 學習摘要與迷思解析", lines=10, interactive=False)

    submit_button.click(
        process_grades_and_summary,
        inputs=grade_input,
        outputs=[sheet_output, summary_output, plot_output]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e46b9960f3a955444.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
